# 05 — Visualization

**Inputs needed:** `results/training_history_*.json`, `results/evaluation_metrics.json`, `attention/*_attention.nii.gz`, processed CT volumes.
**Outputs produced:** PNG plots under `Plots/` plus inline figures.
**Runtime:** Seconds (pure plotting).


Render the artifacts produced by Phase 4 + 6:

- Training curves from `results/training_history_*.json`.
- ROC comparison across radiomics / SwinViT / fused.
- Ablation bar chart over `results/evaluation_metrics.json`.
- SwinViT attention heatmap overlay on the original CT.
- Per-patient cropped slice viewer.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

from src.utils.config import load_config
from src.utils.visualization import (
    plot_ablation_comparison,
    plot_roc_curve,
    plot_slice_overlay,
    plot_training_curves,
)

cfg = load_config(ROOT / "configs" / "default.yaml")
results_dir = Path(cfg["paths"]["results_dir"])
plots_dir = Path(cfg["paths"]["plots_dir"])
attention_dir = Path(cfg["paths"]["attention_dir"])
processed_dir = Path(cfg["paths"]["processed_dir"])

In [ ]:
plot_training_curves(
    {
        "swinvit": results_dir / "training_history_swinvit.json",
        "fusion": results_dir / "training_history_fusion.json",
    },
    plots_dir / "training_curves.png",
)
from IPython.display import Image
Image(filename=str(plots_dir / "training_curves.png"))

In [ ]:
metrics_path = results_dir / "evaluation_metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    plot_ablation_comparison(metrics, plots_dir / "ablation_bar_chart.png")
    Image(filename=str(plots_dir / "ablation_bar_chart.png"))
else:
    print("Run scripts/run_evaluation.py first to produce evaluation_metrics.json")

In [ ]:
import pandas as pd

scores_path = results_dir / "baseline_scores.csv"
if scores_path.exists() and metrics_path.exists():
    baseline_df = pd.read_csv(scores_path)
    score_dict = {
        "radiomics": baseline_df["score"].tolist(),
    }
    plot_roc_curve(
        baseline_df["label"].tolist(),
        score_dict,
        plots_dir / "roc_radiomics.png",
    )
    Image(filename=str(plots_dir / "roc_radiomics.png"))
else:
    print("Baseline scores missing — run scripts/run_radiomics.py first.")

## SwinViT attention heatmap overlay

In [ ]:
import seaborn as sns  # noqa: F401

heatmaps = sorted(attention_dir.glob("*_attention.nii.gz"))
print("Available heatmaps:", [p.name for p in heatmaps[:10]])
if heatmaps:
    pid = heatmaps[0].stem.replace("_heatmap.nii", "")
    ct = nib.load(str(processed_dir / pid / "before.nii.gz")).get_fdata()
    heat = nib.load(str(heatmaps[0])).get_fdata()
    z = np.unravel_index(np.argmax(heat), heat.shape)[-1]
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(ct[..., z].T, cmap="gray", origin="lower")
    ax[0].set_title(f"CT slice {z}")
    ax[0].axis("off")
    ax[1].imshow(ct[..., z].T, cmap="gray", origin="lower")
    ax[1].imshow(heat[..., z].T, cmap="jet", alpha=0.45, origin="lower")
    ax[1].set_title("Attention overlay")
    ax[1].axis("off")
    plt.tight_layout(); plt.show()

## Slice viewer with mask overlay

In [ ]:
patient_dirs = [p for p in processed_dir.iterdir() if (p / "before.nii.gz").exists()]
if patient_dirs:
    pdir = patient_dirs[0]
    vol = nib.load(str(pdir / "before.nii.gz")).get_fdata()
    mask = nib.load(str(pdir / "before_liver.nii.gz")).get_fdata()
    plot_slice_overlay(vol, mask, slice_idx=vol.shape[-1] // 2)